In [1]:
from google.colab import files
uploaded = files.upload()

Saving pakwheels_cleaned.csv to pakwheels_cleaned.csv


In [3]:
import pandas as pd

df = pd.read_csv("pakwheels_cleaned.csv")
print(df.shape)

(47665, 12)


In [4]:
df = df.drop(columns=["color"])

top_cities = df["city"].value_counts().head(15).index
df["city"] = df["city"].where(df["city"].isin(top_cities), "Other")

print(df["city"].value_counts())

city
Lahore        10661
Karachi        9029
Islamabad      7612
Other          7470
Rawalpindi     3062
Peshawar       2180
Faisalabad     1675
Multan         1313
Gujranwala     1165
Sialkot         899
Sargodha        550
Abbottabad      458
Hyderabad       447
Mardan          387
Quetta          385
Bahawalpur      372
Name: count, dtype: int64


In [5]:
print(df["registered"].value_counts())
print()
print("Total unique values:", df["registered"].nunique())

registered
Islamabad      11394
Lahore         10558
Punjab          6207
Sindh           6120
Karachi         6085
               ...  
Khushab            1
Sibi               1
Jaranwala          1
Swatmingora        1
Dunia Pur          1
Name: count, Length: 94, dtype: int64

Total unique values: 94


In [6]:
df["registered"] = df["registered"].where(df["registered"] == "Un-Registered", "Registered")

print(df["registered"].value_counts())

registered
Registered       42952
Un-Registered     4713
Name: count, dtype: int64


In [7]:
top_models = df["car_model"].value_counts().head(30).index
df["car_model"] = df["car_model"].where(df["car_model"].isin(top_models), "Other")

print(df["car_model"].value_counts())

car_model
Other       10112
Corolla      7271
Civic        4627
City         3353
Mehran       2682
Alto         2484
Cultus       2296
Wagon        1324
Vitz         1234
Swift         940
Yaris         921
Vezel         796
Mira          749
Sportage      741
Passo         698
Prado         662
Bolan         655
Prius         654
Hilux         593
Land          579
Aqua          576
Fortuner      469
Benz          453
Cuore         411
Every         377
Dayz          376
N             373
Khyber        330
BR-V          325
Tucson        305
Move          299
Name: count, dtype: int64


In [8]:
df = pd.get_dummies(df, columns=["city", "fuel_type", "transmission",
                                   "registered", "assembly",
                                   "brand", "car_model"], drop_first=True)

print(df.shape)
df.head()

(47665, 116)


,price,year,mileage,engine_capacity,city_Bahawalpur,city_Faisalabad,city_Gujranwala,city_Hyderabad,city_Islamabad,city_Karachi,...,car_model_Passo,car_model_Prado,car_model_Prius,car_model_Sportage,car_model_Swift,car_model_Tucson,car_model_Vezel,car_model_Vitz,car_model_Wagon,car_model_Yaris
0,2650000,2014,82000,660,False,False,False,False,False,False,...,False,False,False,False,False,False,False,False,False,False
1,5400000,2020,59000,1200,False,False,False,False,False,False,...,False,False,False,False,False,False,False,False,False,False
2,7850000,2021,41000,1500,False,False,False,False,False,False,...,False,False,False,False,False,False,False,False,False,True
3,10700000,2017,37000,1500,False,False,False,False,True,False,...,False,False,False,False,False,False,False,False,False,False
4,3600000,2016,45000,1500,False,False,False,False,False,True,...,False,False,False,False,False,False,False,False,False,False


In [9]:
from sklearn.model_selection import train_test_split

X = df.drop("price", axis=1)
y = df["price"]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print("Training set:", X_train.shape)
print("Testing set:", X_test.shape)

Training set: (38132, 115)
Testing set: (9533, 115)


In [10]:
from sklearn.linear_model import LinearRegression

model = LinearRegression()
model.fit(X_train, y_train)

print("Model train ho gaya!")

Model train ho gaya!


In [11]:
from sklearn.metrics import mean_absolute_error, r2_score

predictions = model.predict(X_test)

mae = mean_absolute_error(y_test, predictions)
r2 = r2_score(y_test, predictions)

print(f"Mean Absolute Error: {mae:,.0f} PKR")
print(f"R² Score: {r2:.3f}")

Mean Absolute Error: 1,600,915 PKR
R² Score: 0.599


In [12]:
from sklearn.ensemble import RandomForestRegressor

rf_model = RandomForestRegressor(n_estimators=100, random_state=42)
rf_model.fit(X_train, y_train)

rf_predictions = rf_model.predict(X_test)

rf_mae = mean_absolute_error(y_test, rf_predictions)
rf_r2 = r2_score(y_test, rf_predictions)

print(f"Random Forest - Mean Absolute Error: {rf_mae:,.0f} PKR")
print(f"Random Forest - R² Score: {rf_r2:.3f}")

Random Forest - Mean Absolute Error: 443,648 PKR
Random Forest - R² Score: 0.937


In [13]:
# Ek sample car define karein - values apni marzi se badal sakti hain
sample = pd.DataFrame([{
    "year": 2018,
    "mileage": 60000,
    "engine_capacity": 1300,
    "city_Lahore": 1,
    "fuel_type_Petrol": 1,
    "transmission_Manual": 0,   # 0 matlab Automatic
    "registered_Un-Registered": 0,   # 0 matlab Registered
    "assembly_Local": 0,   # 0 matlab Imported
    "brand_Toyota": 1,
    "car_model_Corolla": 1,
}])

# Baaki sab columns automatically 0 ban jayengi, X training data jaisi order mein
sample = sample.reindex(columns=X_train.columns, fill_value=0)

predicted_price = rf_model.predict(sample)[0]
print(f"Predicted price: {predicted_price:,.0f} PKR")

Predicted price: 3,828,100 PKR


In [14]:
sample2 = pd.DataFrame([{
    "year": 2023,
    "mileage": 37213,
    "engine_capacity": 660,
    "city_Lahore": 0,             # koi specific city nahi batayi, is liye 0 (matlab "Other" city)
    "fuel_type_Hybrid": 1,
    "transmission_Manual": 0,     # 0 matlab Automatic
    "registered_Un-Registered": 0,  # 0 matlab Registered
    "assembly_Local": 0,          # 0 matlab Imported
    "brand_Nissan": 1,
    "car_model_Dayz": 1,
}])

sample2 = sample2.reindex(columns=X_train.columns, fill_value=0)

predicted_price2 = rf_model.predict(sample2)[0]
print(f"Predicted price: {predicted_price2:,.0f} PKR")

Predicted price: 3,677,260 PKR


In [15]:
sample3 = pd.DataFrame([{
    "year": 2023,
    "mileage": 37213,
    "engine_capacity": 660,
    "city_Multan": 1,
    "fuel_type_Hybrid": 1,
    "transmission_Manual": 0,
    "registered_Un-Registered": 0,
    "assembly_Local": 0,
    "brand_Nissan": 1,
    "car_model_Dayz": 1,
}])

sample3 = sample3.reindex(columns=X_train.columns, fill_value=0)

predicted_price3 = rf_model.predict(sample3)[0]
print(f"Predicted price (Multan): {predicted_price3:,.0f} PKR")

Predicted price (Multan): 3,730,660 PKR


In [16]:
sample4 = pd.DataFrame([{
    "year": 2025,
    "mileage": 60000,
    "engine_capacity": 1500,
    "city_Multan": 1,
    "fuel_type_Petrol": 1,
    "transmission_Manual": 0,
    "registered_Un-Registered": 0,    # Punjab mein registered hai, so "Registered"
    "assembly_Local": 1,              # ab Local
    "brand_Honda": 1,
    "car_model_Other": 1,
}])

sample4 = sample4.reindex(columns=X_train.columns, fill_value=0)

predicted_price4 = rf_model.predict(sample4)[0]
print(f"Predicted price: {predicted_price4:,.0f} PKR")

Predicted price: 6,898,510 PKR


In [17]:
sample5 = pd.DataFrame([{
    "year": 2009,
    "mileage": 168286,
    "engine_capacity": 1800,
    "city_Islamabad": 1,
    "fuel_type_Petrol": 1,
    "transmission_Manual": 0,
    "registered_Un-Registered": 0,
    "assembly_Local": 1,
    "brand_Honda": 1,
    "car_model_Civic": 1,
}])

sample5 = sample5.reindex(columns=X_train.columns, fill_value=0)

predicted_price5 = rf_model.predict(sample5)[0]
print(f"Predicted price: {predicted_price5:,.0f} PKR")

Predicted price: 2,637,100 PKR
